# Lecture 29 — LSTM and GRU: Memory With Gates
**NAVTTC / NIAI — AI/ML Trainer Cohort · Mansab Ali**

Six short exercises. Each has a `# YOUR CODE` area and a check cell that prints `OK` or tells you what to fix.
Run the cells top to bottom.

In [ ]:
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"]="3"
import numpy as np, tensorflow as tf
from tensorflow import keras
L = keras.layers
def sigmoid(z): return 1/(1+np.exp(-z))
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ")+name+("" if ok else "  -> "+hint))

## Exercise 1 — Gates are numbers between 0 and 1
Compute sigma(z) for z = -4, 0, 4. Slide 4 has the values to compare.

In [ ]:
z = np.array([-4.0, 0.0, 4.0])
gates = None   # YOUR CODE: sigmoid of z

In [ ]:
check('gate values', gates is not None and np.allclose(gates,[0.018,0.5,0.982],atol=1e-3), 'expected 0.018, 0.5, 0.982')

## Exercise 2 — One LSTM step by hand (slides 7-10)
Weights from slide 7. Fill in the step function, then run it on x = 1, 2, 3. Start with c = 0, h = 0.

Weights (on x, on h_old, bias): forget (0.5, 0.5, 1.0); input, candidate and output (1, 0, 0).

In [ ]:
W = {"f":(0.5,0.5,1.0), "i":(1,0,0), "g":(1,0,0), "o":(1,0,0)}
def lstm_step(x, h_old, c_old):
    # YOUR CODE: compute f, i, g, o, then c and h
    f = i = g = o = None
    c = h = None
    return h, c, (f, i, g, o)

In [ ]:
h, c = 0.0, 0.0; hs, cs = [], []
for x in (1, 2, 3):
    h, c, _ = lstm_step(x, h, c); hs.append(h); cs.append(c)
    print(f"x={x}  c={c:.3f}  h={h:.3f}")
check('h = 0.370, 0.770, 0.931', np.allclose(hs,[0.370,0.770,0.931],atol=1e-3), 'compare with slide 10')
check('c = 0.557, 1.350, 2.226', np.allclose(cs,[0.557,1.350,2.226],atol=1e-3), 'c_new = f*c_old + i*g')

## Exercise 3 — Count the parameters
Count by formula (u = units, d = input features), then check with `model.summary()`. For 16 units and 1 input feature the plain RNN has 288. Keras GRU keeps two bias vectors, so it uses 3 x (u*d + u*u + 2u).

In [ ]:
u, d = 16, 1
p_rnn = p_gru = p_lstm = None   # YOUR CODE: formulas

In [ ]:
real = {}
for name, C in (("rnn", L.SimpleRNN), ("gru", L.GRU), ("lstm", L.LSTM)):
    m = keras.Sequential([L.Input((10, 1)), C(16)]); real[name] = m.count_params()
print(real)
check('SimpleRNN', p_rnn == real['rnn'])
check('GRU', p_gru == real['gru'])
check('LSTM', p_lstm == real['lstm'])
check('LSTM(32) on 1 feature', 4*(32*1+32*32+32) == keras.Sequential([L.Input((10,1)), L.LSTM(32)]).count_params())

## Exercise 4 — Shapes of the outputs
Run an LSTM on a batch of 5 sequences, 7 steps, 3 features, with 8 units. Fill in the three expected shapes (as tuples), then check.

In [ ]:
x = np.random.RandomState(0).randn(5, 7, 3).astype('float32')
shape_last = shape_seq = shape_state = None   # YOUR CODE: expected tuples

In [ ]:
out = L.LSTM(8)(x)
seq = L.LSTM(8, return_sequences=True)(x)
o, h, c = L.LSTM(8, return_state=True)(x)
print(out.shape, seq.shape, h.shape, c.shape)
check('last output', tuple(out.shape) == shape_last)
check('all steps', tuple(seq.shape) == shape_seq)
check('h and c', tuple(h.shape) == shape_state == tuple(c.shape))
og, hg = L.GRU(8, return_state=True)(x)
check('GRU returns only h (no c)', tuple(hg.shape) == (5, 8))

## Exercise 5 — Does the gate help? A memory test
Task: the label is the sign of the FIRST value of a noisy sequence of length T. The model must remember it to the end.

Write `run(cell, T, seed)` that builds `Input -> cell(16) -> Dense(1, sigmoid)`, trains 20 epochs (Adam 0.005, clipnorm 1.0, batch 64) and returns test accuracy. Then compare SimpleRNN, GRU and LSTM on **several seeds**. Use T = 20 here so it runs quickly.

In [ ]:
def make_data(N, T, seed):
    r = np.random.RandomState(seed); x = r.randn(N, T, 1).astype('float32')
    return x, (x[:, 0, 0] > 0).astype('float32')

def run(Cell, T, seed, epochs=20):
    # YOUR CODE: set the seed, build, compile, fit, return test accuracy
    return None

In [ ]:
results = {}
for name, C in (("SimpleRNN", L.SimpleRNN), ("GRU", L.GRU), ("LSTM", L.LSTM)):
    results[name] = [run(C, 20, s) for s in range(3)]
    print(f"{name:9s}", np.round(results[name], 3), " mean", round(float(np.mean(results[name])), 3))
check('run() returns accuracies', all(r is not None for v in results.values() for r in v), 'return the test accuracy')
check('every cell beats a coin flip on average', all(np.mean(v) > 0.6 for v in results.values()), 'check labels and training')

## Exercise 6 — Reflect (write 3-4 sentences)
1. Look at your Exercise 5 numbers. Did one cell win in every seed? What would you need before saying LSTM is better?
2. Why does a decoder that starts from an LSTM encoder need *two* states (h and c)?
3. For your own project, which cell would you try first, and what baseline would you compare against?

**Your answers:**

